In [3]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as t
from geopy.distance import geodesic
from math import sqrt
import pandas as pd

spark = SparkSession.builder.getOrCreate()
spark

data_trips = spark.read.format('csv').option('header', 'true').load('trip.csv')
data_stations = spark.read.format('csv').option('header', 'true').load('station.csv')

data_trips.show(5)
data_stations.show(5)

# Приводим типы данных для trips
data_trips = data_trips.withColumn("duration", F.col("duration").cast(t.IntegerType()))

# Приводим типы данных для stations
data_stations = data_stations.withColumn("lat", F.col("lat").cast(t.DoubleType())) \
                             .withColumn("long", F.col("long").cast(t.DoubleType()))

+----+--------+---------------+--------------------+----------------+---------------+--------------------+--------------+-------+-----------------+--------+
|  id|duration|     start_date|  start_station_name|start_station_id|       end_date|    end_station_name|end_station_id|bike_id|subscription_type|zip_code|
+----+--------+---------------+--------------------+----------------+---------------+--------------------+--------------+-------+-----------------+--------+
|4576|      63|8/29/2013 14:13|South Van Ness at...|              66|8/29/2013 14:14|South Van Ness at...|            66|    520|       Subscriber|   94127|
|4607|      70|8/29/2013 14:42|  San Jose City Hall|              10|8/29/2013 14:43|  San Jose City Hall|            10|    661|       Subscriber|   95138|
|4130|      71|8/29/2013 10:16|Mountain View Cit...|              27|8/29/2013 10:17|Mountain View Cit...|            27|     48|       Subscriber|   97214|
|4251|      77|8/29/2013 11:29|  San Jose City Hall|      

### 1. Найти велосипед с максимальным временем пробега ###  
Группируем данные по bike_id, суммируем время всех поездок и сортируем по убыванию.

In [4]:
# Находим велосипед с максимальной суммарной длительностью
bike_max_duration_row = data_trips.groupBy("bike_id") \
    .agg(F.sum("duration").alias("total_duration")) \
    .orderBy(F.desc("total_duration")) \
    .first()

top_bike_id = bike_max_duration_row["bike_id"]
print(f"Велосипед с максимальным временем пробега: {top_bike_id} (Время: {bike_max_duration_row['total_duration']} сек.)")

Велосипед с максимальным временем пробега: 535 (Время: 18611693 сек.)


### 2. Найти наибольшее геодезическое расстояние между станциями ###  
В station.csv всего около 70 станций, поэтому мы можем безопасно использовать crossJoin (декартово произведение), чтобы получить все возможные пары станций. Для расчета расстояния напишем UDF (User Defined Function) с использованием библиотеки geopy.

In [ ]:
import itertools

stations_list = data_stations.select("name", "lat", "long").dropna().collect()

max_dist = 0
pair = (None, None)

for s1, s2 in itertools.combinations(stations_list, 2):
    dist = geodesic((s1["lat"], s1["long"]), (s2["lat"], s2["long"])).kilometers
    if dist > max_dist:
        max_dist = dist
        pair = (s1["name"], s2["name"])

print(f"Наибольшее расстояние: {max_dist:.2f} км")
print(f"Между станциями: {pair[0]} и {pair[1]}")

Наибольшее расстояние: 69.92 км
Между станциями: SJSU - San Salvador at 9th и Embarcadero at Sansome


### 3. Найти путь велосипеда с максимальным временем пробега  
Используем top_bike_id, который мы нашли в первой задаче. Фильтруем поездки по этому велосипеду и сортируем их по времени начала. Примечание: формат даты в CSV обычно M/d/yyyy H:mm, его нужно правильно распарсить для точной сортировки.

In [7]:
# Фильтруем датафрейм, преобразуем строку даты в Timestamp для правильной сортировки хронологии
bike_path = data_trips.filter(F.col("bike_id") == top_bike_id) \
    .withColumn("start_timestamp", F.to_timestamp("start_date", "M/d/yyyy H:mm")) \
    .orderBy("start_timestamp") \
    .select("start_timestamp", "start_station_name", "end_station_name")

print(f"Путь велосипеда {top_bike_id}:")
bike_path.show(truncate=False)

Путь велосипеда 535:
+-------------------+---------------------------------------------+----------------------------------------+
|start_timestamp    |start_station_name                           |end_station_name                        |
+-------------------+---------------------------------------------+----------------------------------------+
|2013-08-29 19:32:00|Post at Kearney                              |San Francisco Caltrain (Townsend at 4th)|
|2013-08-29 21:38:00|San Francisco Caltrain (Townsend at 4th)     |San Francisco Caltrain 2 (330 Townsend) |
|2013-08-30 08:40:00|San Francisco Caltrain 2 (330 Townsend)      |Market at Sansome                       |
|2013-08-30 09:10:00|Market at Sansome                            |2nd at South Park                       |
|2013-09-01 12:58:00|2nd at Townsend                              |Davis at Jackson                        |
|2013-09-05 11:59:00|San Francisco City Hall                      |Civic Center BART (7th at Market)       

### 4. Найти количество велосипедов в системе  
Здесь всё просто: считаем количество уникальных bike_id в таблице поездок.

In [8]:
total_bikes = data_trips.select("bike_id").distinct().count()
print(f"Общее количество велосипедов в системе: {total_bikes}")

Общее количество велосипедов в системе: 700


### 5. Найти пользователей, потративших на поездки более 3 часов  
Важный нюанс: В открытом датасете велопарковок Сан-Франциско нет уникального идентификатора пользователя (user_id). У нас есть только тип подписки (subscription_type) и почтовый индекс (zip_code). Поэтому мы не можем сгруппировать все поездки одного конкретного человека. Но мы можем интерпретировать задачу как "Найти конкретные поездки (сессии), которые длились более 3 часов". 3 часа = 10800 секунд (длительность в датасете указана в секундах).

In [9]:
# Находим поездки длительностью строго больше 10800 секунд
long_trips = data_trips.filter(F.col("duration") > 10800) \
    .select("id", "duration", "start_date", "subscription_type", "zip_code")

print("Поездки (пользователи), длившиеся более 3 часов:")
long_trips.show()

Поездки (пользователи), длившиеся более 3 часов:
+----+--------+---------------+-----------------+--------+
|  id|duration|     start_date|subscription_type|zip_code|
+----+--------+---------------+-----------------+--------+
|4639|   11118|8/29/2013 15:18|         Customer|    NULL|
|4637|   11272|8/29/2013 15:17|         Customer|    NULL|
|4528|   12280|8/29/2013 13:39|       Subscriber|   94536|
|4363|   15244|8/29/2013 12:23|         Customer|    NULL|
|4193|   18192|8/29/2013 11:04|         Customer|   72150|
|4190|   18240|8/29/2013 11:04|         Customer|   72150|
|4225|   21612|8/29/2013 11:18|         Customer|   58553|
|4663|   52698|8/29/2013 15:34|       Subscriber|   94301|
|4532|   84990|8/29/2013 13:43|         Customer|   94118|
|4521|   85385|8/29/2013 13:37|         Customer|   94118|
|5069|   86102|8/29/2013 21:41|         Customer|   94111|
|4505|   97713|8/29/2013 13:30|       Subscriber|   94039|
|5539|   10805|8/30/2013 12:32|         Customer|   94133|
|6032| 